# ZADANIE: Model Card Tensor

- przygotuj dataframe w oparciu o specyfikacje "model cards" dla poszczególnych modeli

# DOCS

- [dokumentacja pliku HF:`config.json`](https://huggingface.co/docs/transformers/main_classes/configuration)
- model cards:
  1. [Bielik-7B-v0.1](https://huggingface.co/speakleash/Bielik-7B-v0.1)
  2. [Llama-3.1-8B](https://huggingface.co/meta-llama/Llama-3.1-8B)
  3. [Mistral-7B-v0.1](https://huggingface.co/mistralai/Mistral-7B-v0.1)
  4. dla ambitnych 🔥 (inna struktura)
    - [DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1)
    - [Qwen2.5-7B](https://huggingface.co/Qwen/Qwen2.5-7B)

In [ ]:
!pip install pandas

In [ ]:
from pathlib import Path
import json
import sys
import os
import pandas as pd

base = Path(os.getcwd())
pattern = "*-config.json"
matches = sorted(base.rglob(pattern))
files = [p.name for p in matches]
# print(files)
# print(json.dumps(files, indent=2))

failing = []
model_cards = []

for p in matches:
    try:
        with p.open('r', encoding='utf-8') as f:
            data = json.load(f)
        model_cards.append({
            'filename': p.name,
            'json': data
        })
    except json.JSONDecodeError:
        failing.append(f"Niepoprawny format JSON (Pusty/Błędny) w pliku: {p.name}")
    except ValueError as e:
        failing.append(f"Błąd danych: {e} Plik: {p.name}")
    except Exception as e:
        failing.append(f"Inny nieznany błąd przy wczytywaniu {p.name}: {e}")

if len(failing):
    print(failing)
else:
    print('All models calrds loaded successfully')

# We define a mapping logic to interpret what each 'tensor' key represents in the config files
def map_tensor_to_config_value(tensor_name, config):
    # Extract basic dimensions
    h = config.get('hidden_size', 0)
    v = config.get('vocab_size', 0)
    i = config.get('intermediate_size', 0)
    
    # Map based on standard LLM weight matrix shapes
    mapping = {
        'embed_tokens.weight': f"[{v}, {h}]",
        'input_layernorm.weight': f"[{h}]",
        'mlp.down_proj.weight': f"[{h}, {i}]",
        'mpl.gate_proj.weight': f"[{i}, {h}]",
        'mpl.up_proj.weight': f"[{i}, {h}]",
        'post_attention_layernorm.weight': f"[{h}]",
        'self_attn.k_proj.weight': f"[{h}, {h}]",
        'self_attn.o_proj.weight': f"[{h}, {h}]",
        'self_attn.q_proj.weight': f"[{h}, {h}]",
        'self_attn.v_proj.weight': f"[{h}, {h}]",
    }
    return mapping.get(tensor_name, "N/A")

# Build the dataset using only pandas processing
results = []
for item in model_cards:
    name = item['filename'].replace('-config.json', '')
    config = item['json']
    
    row = {'model_name': name}
    for t in tensors:
        row[t] = map_tensor_to_config_value(t, config)
    results.append(row)

df_mapped = pd.DataFrame(results).set_index('model_name')
display(df_mapped.T)



All models calrds loaded successfully


,filename,model_type
0,Bielik-7B-Instruct-v0.1-config.json,mistral
1,DeepSeek-R1-config.json,deepseek_v3
2,Llama-3.1-8B-config.json,llama
3,Mistral-7B-v0.1-config.json,mistral
4,Qwen2.5-7B-Instruct-config.json,qwen2


In [30]:
import pandas as pd

def generate_blank(c):
    return ['[?, ?]' for _ in range(c)]

data = {
    'Bielik-7B-Instruct-v0.1': generate_blank(10),
    'Llama-3.1-8B': generate_blank(10),
    'Mistral-7B-v0.1': generate_blank(10),
    # 'DeepSeek-R1': generate_blank(10),
    # 'Qwen2.5-7B': generate_blank(10),
}

tensors = [
    'embed_tokens.weight',
    'input_layernorm.weight',
    'mlp.down_proj.weight',
    'mpl.gate_proj.weight',
    'mpl.up_proj.weight',
    'post_attention_layernorm.weight',
    'self_attn.k_proj.weight',
    'self_attn.o_proj.weight',
    'self_attn.q_proj.weight',
    'self_attn.v_proj.weight',
]

df = pd.DataFrame(data, index=tensors)

display(df)
# display(df.T) # transpozycja (obrócenie)



,Bielik-7B-Instruct-v0.1,Llama-3.1-8B,Mistral-7B-v0.1
embed_tokens.weight,"[?, ?]","[?, ?]","[?, ?]"
input_layernorm.weight,"[?, ?]","[?, ?]","[?, ?]"
mlp.down_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
mpl.gate_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
mpl.up_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
post_attention_layernorm.weight,"[?, ?]","[?, ?]","[?, ?]"
self_attn.k_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
self_attn.o_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
self_attn.q_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
self_attn.v_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
